# Example 11.1 / 1: A convex allocation QP

Companion notebook for **11.1 Quadratic Models** in the AMPL lab Documentation, section "A convex allocation QP". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A convex allocation QP

The model allocates a fixed total among activities while penalizing squared deviation from targets. The penalty coefficients are positive, so the Hessian is positive definite and the objective is strictly convex; with a nonempty linear feasible set, the optimizer is unique.

![Elliptical contours of a convex quadratic objective touching a linear feasible segment at the constrained optimum](/assets/docs/quadratic-contours.svg)

*Elliptical level curves are characteristic of a positive-definite quadratic. The constrained optimum is the first contour that touches the feasible set.*

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ACT;
param target {ACT};
param weight {ACT} > 0;
param total >= 0;

var Allocate {ACT} >= 0;

minimize WeightedDeviation:
    sum {a in ACT} weight[a] * (Allocate[a] - target[a])^2;

subject to TotalAllocation:
    sum {a in ACT} Allocate[a] = total;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ACT"] = ["A", "B", "C"]
ampl.param["target"] = {"A": 4, "B": 3, "C": 2}
ampl.param["weight"] = {"A": 1, "B": 2, "C": 4}
ampl.param["total"] = 12

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["Allocate"].to_dict())
print(round(ampl.obj["WeightedDeviation"].value(), 6))

HiGHS 1.14.0: optimal solution; objective 5.142857143
0 simplex iterations
0 barrier iterations
solved
{'A': 5.714285664285716, 'B': 3.857142878571428, 'C': 2.428571457142857}
5.142857


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ACT := A B C;
param target := A 4 B 3 C 2;
param weight := A 1 B 2 C 4;
param total := 12;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Allocate, WeightedDeviation;

HiGHS 1.14.0: optimal solution; objective 5.142857143
0 simplex iterations
0 barrier iterations
Allocate [*] :=
A  5.71429
B  3.85714
C  2.42857
;

WeightedDeviation = 5.14286

